# FloodSense: Model Development & Experimental Comparison
**Case Study 74: Flood Risk Analysis**  
**Task:** Supervised Regression on 100k Sample (80k Train, 20k Test)

This notebook evaluates and compares 5 model families under rigorous 5-fold cross-validation on the training set, followed by holdout test evaluation:
1. **Linear Regression** (Baseline & scaling benchmark)
2. **Ridge Regression** (L2 regularization & scaling benchmark)
3. **HistGradientBoosting** (Tree-based gradient boosting)
4. **Random Forest** (Bagged tree ensemble)
5. **Multi-Layer Perceptron (MLP)** (Small neural network)

### Systematic Experiments:
- **Experiment A:** Scaled vs. Unscaled inputs on linear/regularized models.
- **Experiment B:** Original 20 features vs. Original + 3 Engineered Domain Aggregates.
- **Experiment C:** Default vs. Hyperparameter-tuned models (`RandomizedSearchCV`).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_validate, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sys.path.append("..")
from src.preprocess import load_data, split_data, compute_tier_cutoffs, add_engineered_features, build_pipeline

# Load 100k processed sample
df = load_data("../data/processed/sample_100k.csv")
X_train, X_test, y_train, y_test = split_data(df, test_size=0.20, random_state=42)
cutoffs = compute_tier_cutoffs(y_train, save_path="../models/tier_cutoffs.json")

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Test set:     X={X_test.shape}, y={y_test.shape}")
print(f"Tier cutoffs: {cutoffs}")


Training set: X=(80000, 20), y=(80000,)
Test set:     X=(20000, 20), y=(20000,)
Tier cutoffs: {'q33': 0.48, 'q67': 0.525, 'labels': ['Low', 'Medium', 'High'], 'description': 'Quantile-based regional risk tiers calculated on training split.'}


## 1. Feature Engineering Setup

We prepare an alternative feature matrix containing 3 domain aggregate metrics:
- `indicator_mean`: Overall average vulnerability burden.
- `env_mean`: Average across the 8 environmental & hydrological indicators.
- `infra_mean`: Average across the 12 infrastructure, demographic, and governance indicators.


In [ ]:
X_train_eng = add_engineered_features(X_train)
X_test_eng = add_engineered_features(X_test)

print(f"Engineered feature columns added: {list(X_train_eng.columns[-3:])}")
print(f"Shape with aggregates: {X_train_eng.shape}")


Engineered feature columns added: ['indicator_mean', 'env_mean', 'infra_mean']
Shape with aggregates: (80000, 23)


## 2. Experimental Results & Comprehensive Model Comparison

All models were evaluated using **5-Fold Cross-Validation** on the training set (80k rows), followed by a single evaluation on the unseen holdout test set (20k rows).

| Model | Category | Scaled | Features | CV R2 Mean | CV R2 Std | CV RMSE | CV MAE | Test R2 | Test RMSE | Test MAE | CV Train Time (s) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Linear Regression (Unscaled, 20 Feat) | Scaling Experiment | False | 20 (Original) | 0.84421 | 0.00224 | 0.0201 | 0.01583 | 0.84958 | 0.01991 | 0.0157 | 1.85 |
| Linear Regression (Scaled, 20 Feat) | Baseline | True | 20 (Original) | 0.84421 | 0.00224 | 0.0201 | 0.01583 | 0.84958 | 0.01991 | 0.0157 | 1.27 |
| Ridge Regression (Scaled, default alpha=1.0) | Regularization | True | 20 (Original) | 0.84421 | 0.00224 | 0.0201 | 0.01583 | 0.84958 | 0.01991 | 0.0157 | 1.16 |
| Ridge Regression (Unscaled, default alpha=1.0) | Scaling Experiment | False | 20 (Original) | 0.84421 | 0.00224 | 0.0201 | 0.01583 | 0.84958 | 0.01991 | 0.0157 | 0.21 |
| HistGradientBoosting (Default, 20 Feat) | Ensemble Non-Linear | False | 20 (Original) | 0.75598 | 0.00162 | 0.02515 | 0.02048 | 0.76422 | 0.02493 | 0.02039 | 1.29 |
| HistGradientBoosting (150 iters, lr=0.08) | Ensemble Non-Linear | False | 20 (Original) | 0.78277 | 0.00174 | 0.02373 | 0.01926 | 0.78979 | 0.02354 | 0.01913 | 1.75 |
| Random Forest (50 Trees, max_depth=10) | Ensemble Non-Linear | False | 20 (Original) | 0.43933 | 0.00174 | 0.03812 | 0.03107 | 0.43646 | 0.03854 | 0.0315 | 6.66 |
| MLP Neural Net (Scaled, 32-16 layers) | Neural Network | True | 20 (Original) | 0.84457 | 0.0025 | 0.02007 | 0.01575 | 0.85215 | 0.01974 | 0.01554 | 3.16 |
| Linear Regression (Scaled, 20 + 3 Aggregates) | Feature Engineering | True | 23 (Engineered) | 0.84421 | 0.00224 | 0.0201 | 0.01583 | 0.84958 | 0.01991 | 0.0157 | 0.35 |
| HistGradientBoosting (20 + 3 Aggregates) | Feature Engineering | False | 23 (Engineered) | 0.86503 | 0.00191 | 0.0187 | 0.01461 | 0.87092 | 0.01845 | 0.01441 | 0.88 |
| Ridge Regression (Scaled, 20 + 3 Aggregates) | Feature Engineering | True | 23 (Engineered) | 0.84421 | 0.00224 | 0.0201 | 0.01583 | 0.84958 | 0.01991 | 0.0157 | 0.32 |
| Tuned Ridge (alpha=0.207) | Hyperparameter Tuning | True | 20 (Original) | 0.84421 | 0.00224 | 0.0201 | 0.01583 | 0.84958 | 0.01991 | 0.0157 | 1.78 |
| Tuned HistGradientBoosting | Hyperparameter Tuning | False | 20 (Original) | 0.7976 | 0.00257 | 0.02291 | 0.01835 | 0.80573 | 0.02263 | 0.01809 | 8.54 |


In [ ]:
# Load and display the results table
results_path = "../reports/results/model_comparison.csv"
df_results = pd.read_csv(results_path)
df_results.sort_values(by="CV R2 Mean", ascending=False).reset_index(drop=True)


Model,Category,Scaled,Features,CV R2 Mean,CV R2 Std,CV RMSE,CV MAE,Test R2,Test RMSE,Test MAE,CV Train Time (s)
"Linear Regression (Unscaled, 20 Feat)",Scaling Experiment,False,20 (Original),0.84421,0.00224,0.02010,0.01583,0.84958,0.01991,0.01570,1.85
"Linear Regression (Scaled, 20 Feat)",Baseline,True,20 (Original),0.84421,0.00224,0.02010,0.01583,0.84958,0.01991,0.01570,1.27
"Ridge Regression (Scaled, default alpha=1.0)",Regularization,True,20 (Original),0.84421,0.00224,0.02010,0.01583,0.84958,0.01991,0.01570,1.16
"Ridge Regression (Unscaled, default alpha=1.0)",Scaling Experiment,False,20 (Original),0.84421,0.00224,0.02010,0.01583,0.84958,0.01991,0.01570,0.21
"HistGradientBoosting (Default, 20 Feat)",Ensemble Non-Linear,False,20 (Original),0.75598,0.00162,0.02515,0.02048,0.76422,0.02493,0.02039,1.29
"HistGradientBoosting (150 iters, lr=0.08)",Ensemble Non-Linear,False,20 (Original),0.78277,0.00174,0.02373,0.01926,0.78979,0.02354,0.01913,1.75
"Random Forest (50 Trees, max_depth=10)",Ensemble Non-Linear,False,20 (Original),0.43933,0.00174,0.03812,0.03107,0.43646,0.03854,0.03150,6.66
"MLP Neural Net (Scaled, 32-16 layers)",Neural Network,True,20 (Original),0.84457,0.00250,0.02007,0.01575,0.85215,0.01974,0.01554,3.16
"Linear Regression (Scaled, 20 + 3 Aggregates)",Feature Engineering,True,23 (Engineered),0.84421,0.00224,0.02010,0.01583,0.84958,0.01991,0.01570,0.35
HistGradientBoosting (20 + 3 Aggregates),Feature Engineering,False,23 (Engineered),0.86503,0.00191,0.01870,0.01461,0.87092,0.01845,0.01441,0.88


## 3. Detailed Analysis of Experimental Hypotheses

### (a) Scaled vs. Unscaled Inputs
- **Linear Regression:** Unscaled $R^2 = 0.84421$ vs. Scaled $R^2 = 0.84421$.
- **Ridge Regression:** Unscaled $R^2 = 0.84421$ vs. Scaled $R^2 = 0.84421$.
- **Observation:** Because Ordinary Least Squares finds the optimal hyperplane regardless of linear feature rescaling when all 20 features already share identical ranges [0, 16], scaling does not change linear regression predictions. However, scaling is retained as best practice for Ridge regularization and Neural Networks.

### (b) Original 20 Features vs. Original + Engineered Aggregates
- **Linear & Ridge Regression:** Performance remained exactly identical ($R^2 = 0.84421$). This makes mathematical sense: linear regression already fits the optimal linear combination of the 20 variables, so adding a linear mean of those same variables introduces perfect collinearity and adds zero new degrees of freedom.
- **HistGradientBoosting:** Performance surged dramatically from **$R^2 = 0.75598$ to $R^2 = 0.86503$** (and Test $R^2 = 0.87092$).
  - *Theoretical Explanation:* Decision trees partition feature space using axis-parallel, orthogonal splits. Learning an additive diagonal boundary like $x_1 + x_2 + \dots + x_{20}$ requires an exponential number of deep staircase splits. By giving the tree the explicit aggregate `indicator_mean`, the primary split immediately captures the global additive trend, allowing secondary splits to model non-linear noise and interactions.

### (c) Hyperparameter Tuning (`RandomizedSearchCV`)
- **Ridge Tuning:** Exploring $\alpha \in [10^{-2}, 10^3]$ identified an optimal $\alpha \approx 0.207$, yielding identical CV $R^2 = 0.84421$. L2 regularization confirms that the 20 features have no collinear instability.
- **HistGradientBoosting Tuning:** Tuning `max_iter`, `learning_rate`, `max_leaf_nodes`, and `min_samples_leaf` improved raw tree performance from $0.75598$ to $0.79760$ on the original 20 features.

### (d) Random Forest vs. Gradient Boosting
- Random Forest (depth 10) achieved only CV $R^2 = 0.43933$. Averaging independent shallow trees failed to approximate the additive linear combination without extreme depth, whereas boosting sequentially corrects residual errors.


## 4. Final Model Selection

We compare the top candidates:
1. **HistGradientBoosting (with 3 Aggregates):** Highest statistical accuracy (**CV $R^2 = 0.86503$, Test $R^2 = 0.87092$, Test RMSE = 0.01845**).
2. **Ridge Regression (Scaled, 20 Features):** Simpler, fully interpretable parametric baseline (**CV $R^2 = 0.84421$, Test $R^2 = 0.84958$, Test RMSE = 0.01991**).

### Academic Justification:
The gradient boosting model with domain aggregates achieves a genuine +2.1% improvement in $R^2$ and reduces error across both RMSE and MAE. By packaging the feature engineering transformer directly into the `scikit-learn Pipeline`, the model accepts the raw 20 indicators from the user and computes the aggregates internally, combining top performance with seamless deployment.

The pipeline is serialized to `../models/final_model.joblib`.
